<a href="https://colab.research.google.com/github/donoftime2018/Mental-Health-Chatbot/blob/cosineSimilarityOfEmbeddings/Phi_3_mini_4k(FAQ_Ds).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q bitsandbytes>=0.46.1
!pip install -U sentence-transformers
!pip install -U torchao
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer
import pandas as pd
import re
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from peft import LoraConfig, TaskType

In [ ]:
sentenceTransformers = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
device = torch.device("cuda")
device

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
)

In [ ]:
def remove_special_tokens(text):
    text = re.sub(r'<\|.*?\|>', '', text)
    return text.strip()

In [ ]:
def preProcessText(text):
  text = text.lower()
  text = re.sub(r'<.*?>', '', text)
  text = re.sub(r'http\S+|www\.\S+', '', text)
  text = re.sub(r'â€™', "'", text)
  text = re.sub(r'([!?.,])\1+', r'\1', text)
  text = re.sub(r'\s+([.,!?])', r'\1', text)
  text = re.sub(r'\s+', ' ', text).strip()
  text = re.sub(r'[^\w\s]', '', text)
  return text


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    "unsloth/Phi-3-mini-4k-instruct",
    # "/content/drive/MyDrive/Colab Notebooks/mental-health-qa-phi3mini4k", #unsloth/Phi-3-mini-4k-instruct
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
)

fineTunedModel = AutoModelForCausalLM.from_pretrained(
    "/content/drive/MyDrive/Colab Notebooks/mental-health-qa-phi3mini4k", #unsloth/Phi-3-mini-4k-instruct
)

model.to(device)
fineTunedModel.to(device)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("unsloth/Phi-3-mini-4k-instruct")
tokenizer.pad_token = tokenizer.eos_token
tokenizer

In [ ]:
messages = [
    # {"role": "assistant", "content": "I feel completely lost after my dog died. What should I do to cope day to day?"},
    {"role": "user", "content": input("")}
]

In [ ]:
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True,
	return_dict=True,
	return_tensors="pt").to(device)
inputs

In [ ]:
# model_inputs = encoded_message.to(device)
# model_inputs

In [ ]:
fineTunedOutputs = fineTunedModel.generate(**inputs, max_new_tokens=150, max_length=50, num_return_sequences=3, do_sample=True)
fineTunedOutputs

In [ ]:
ogOutputs = model.generate(**inputs, max_new_tokens=150, max_length=50, num_return_sequences=3, do_sample=True)
ogOutputs

In [ ]:
print(tokenizer.decode(fineTunedOutputs))
fineTunedResponse = tokenizer.decode(fineTunedOutputs[0])
fineTunedResponse

In [ ]:
print(tokenizer.decode(ogOutputs))
ogResponse = tokenizer.decode(ogOutputs[0])
ogResponse

In [ ]:
encodedFineTuned = sentenceTransformers.encode(fineTunedResponse)
encodedFineTuned.shape

In [ ]:
encodedOg = sentenceTransformers.encode(ogResponse)
encodedOg.shape

In [ ]:
cosine_similarity([encodedFineTuned], [encodedOg])

In [ ]:
dataset = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Mental_Health_FAQ.csv")

In [ ]:
questions = dataset['Questions'].astype("str").apply(preProcessText).apply(remove_special_tokens).values
questions[:1]

In [ ]:
answers = dataset['Answers'].astype("str").apply(preProcessText).apply(remove_special_tokens).values
answers[:1]

In [ ]:
question_ids = dataset['Question_ID'].astype('int').values
question_ids[:1]

In [ ]:
def combineText(example):
  messages = [
      {"role": "user", "content": example['questions']},
      {"role": "assistant", "content": example['answers']}
  ]
  formatted_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
  return {"text": formatted_text}

In [ ]:
def encode(example):
  return tokenizer(example['text'], truncation=True, padding=True, max_length=128)

In [ ]:
def add_labels(example):
    example['labels']=example['input_ids']
    return example

In [ ]:
datasets = Dataset.from_dict({
    "questions": questions,
    "answers": answers
})
datasets

In [ ]:
datasets_split = datasets.train_test_split( test_size=0.2)
datasets_split

In [ ]:
trainSet = datasets_split['train']
trainSet

In [ ]:
testSet = datasets_split['test']
testSet

In [ ]:
trainSet = trainSet.map(combineText)

In [ ]:
testSet = testSet.map(combineText)

In [ ]:
trainSet = trainSet.map(encode, batched=True)
trainSet

In [ ]:
testSet = testSet.map(encode, batched=True)
testSet

In [ ]:
trainSet = trainSet.map(add_labels)
trainSet

In [ ]:
testSet = testSet.map(add_labels)
testSet

In [ ]:
trainingArgs = TrainingArguments(
    output_dir="./output",
    num_train_epochs=2,
    learning_rate=2e-5,
    per_device_train_batch_size=1, # Further reduced batch size to prevent OOM
    per_device_eval_batch_size=1, # Reduced eval batch size for consistency
    gradient_accumulation_steps=8, # Use gradient accumulation to achieve an effective batch size of 1 * 8 = 8
    eval_strategy='steps',
    weight_decay=0.01,
    warmup_steps=20,
    logging_dir=None,
    fp16=True,
    bf16=False, # Use bfloat16 for better memory stability and efficiency on T4
    logging_steps=50,
    gradient_checkpointing=True, # Enable gradient checkpointing to save memory
    max_grad_norm = 1.0,
    report_to="none"
)

In [ ]:
model.add_adapter(lora_config, adapter_name="my_adapter")

In [ ]:
trainer = Trainer(
    model=model,
    args=trainingArgs,
    train_dataset=trainSet,
    eval_dataset=testSet
)

In [ ]:
trainer.evaluate(testSet)

In [ ]:
trainer.predict(testSet)

In [ ]:
trainer.train()

In [ ]:
trainer.save_model("/content/drive/MyDrive/Colab Notebooks/mental-health-qa-phi3mini4k")

In [ ]:
trainer.evaluate(testSet)

In [ ]:
trainer.predict(testSet)